# TransGraph-PPI: Complete Model Training & Evaluation Pipeline
### Model Architecture: ESM-2 150M (`facebook/esm2_t30_150M_UR50D`) + GraphSAGE + XGBoost Stacking

This notebook executes the complete end-to-end training and benchmark pipeline on a T4 GPU:
1. **Environment Setup & GPU Check**
2. **Dependency & Torchvision Fix** (uninstalls conflicting torchvision to load EsmModel cleanly)
3. **Dataset Splits Setup** (restores or uploads `train.csv`, `val.csv`, `test.csv`)
4. **Dataset Verification** (verifies pair-disjoint split integrity)
5. **ESM-2 150M Feature Extraction** (smart skip: restores pre-computed `embeddings.pt` from Drive in seconds!)
6. **Graph Construction** (smart skip: restores `ppi_graph.pt` from Drive in seconds!)
7. **Sequence Model Training** (Residual MLP on 640-d embeddings with early stopping)
8. **Graph Model Training** (GraphSAGE link prediction + Platt calibration)
9. **Random Forest Baseline Training**
10. **5-Fold OOF XGBoost Ensemble Training**
11. **Final Test Evaluation & Comparison** (`final_test_metrics.json`)
12. **External Benchmarks** (Cold-Start, SHS27k, HuRI)
13. **Artifact Packaging & Checksums** (`esm150m_results.zip`)


In [13]:
# Step 1: Verify GPU Acceleration (T4 GPU)
import torch
print("PyTorch Version:", torch.__version__)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"CUDA is ACTIVE! GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("WARNING: CUDA is not active. If in Colab, select Runtime -> Change runtime type -> T4 GPU.")


PyTorch Version: 2.11.0+cu128
CUDA is ACTIVE! GPU: Tesla T4 (15.64 GB VRAM)


In [14]:
# Step 2: Environment Setup & Google Drive Mount (if on Google Colab)
import os
import sys

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("Google Drive mounted.")
    if os.path.exists("/content") and not os.path.exists("/content/Majorproject"):
        !git clone -b metric-optimization https://github.com/AshShenoy14/Majorproject.git /content/Majorproject
        %cd /content/Majorproject
    elif os.path.exists("/content/Majorproject"):
        %cd /content/Majorproject
except ImportError:
    print("Running in local VS Code / Colab Extension environment.")

print("Working directory:", os.getcwd())
sys.path.append(os.getcwd())


/content/Majorproject
Working directory: /content/Majorproject


In [15]:
# Step 3: Fix Dependencies & Eliminate Torchvision Conflicts
# Uninstall torchvision because it triggers C++ operator conflicts with PyTorch on Colab
!pip uninstall -y torchvision
!pip install -q -r requirements.txt
print("Requirements installed successfully. Torchvision removed (clean NLP/graph runtime).")


Requirements installed successfully. Torchvision removed (clean NLP/graph runtime).


In [16]:
# Step 3.5: Load / Restore Dataset Splits (train.csv, val.csv, test.csv)
import os

os.makedirs("data/processed", exist_ok=True)
drive_data = "/content/drive/MyDrive/transgraph_ppi_esm150m/data_processed"

# 1. Try restoring from Google Drive backup if available
if os.path.exists(f"{drive_data}/train.csv"):
    print("Restoring train.csv, val.csv, test.csv from Google Drive...")
    !cp -r "$drive_data"/*.csv data/processed/ 2>/dev/null || true
    print("Splits restored from Google Drive!")

# 2. If not on Drive, prompt for quick upload or generate
if not os.path.exists("data/processed/train.csv"):
    print("Notice: train.csv not found in data/processed/.")
    try:
        from google.colab import files
        print("Please select train.csv, val.csv, test.csv, bio_metadata_cache.csv from your local E:\\majorproject\\data\\processed folder:")
        uploaded = files.upload()
        !mv *.csv data/processed/ 2>/dev/null || true
    except Exception:
        print("Generating splits using preprocess_data.py...")
        !python src/data/preprocess_data.py --seed 42

!ls -lh data/processed/*.csv


Restoring train.csv, val.csv, test.csv from Google Drive...
Splits restored from Google Drive!
-rw------- 1 root root  13K Sep 24 13:25 data/processed/bio_metadata_cache.csv
-rw------- 1 root root 670K Sep 24 13:25 data/processed/test.csv
-rw------- 1 root root 5.3M Sep 24 13:25 data/processed/train.csv
-rw------- 1 root root 670K Sep 24 13:25 data/processed/val.csv


In [17]:
# Step 4: Verify Dataset Splits & Negative Sampling
!python src/data/collect_ppi.py
!python scripts/verify_splits.py
!python scripts/evaluation/audit_measurements.py --out assets/evaluation/audit/audit_after_data.json --skip-model-metrics


STRING file already exists at /content/Majorproject/data/raw/9606.protein.links.v12.0.txt.gz
STRING sequences file already exists at /content/Majorproject/data/raw/9606.protein.sequences.v12.0.fa.gz
Please download the latest BioGRID release for Homo sapiens manually if not present.
Place it in:  /content/Majorproject/data/raw
      DATASET SPLIT & NEGATIVE SAMPLING VERIFICATION

--- Dataset Sizes ---
Train: 161369 rows (Pos: 80685, Neg: 80684)
Val:   20171 rows (Pos: 10085, Neg: 10086)
Test:  20172 rows (Pos: 10086, Neg: 10086)

--- Internal Canonical Duplicates ---
Train internal duplicates: 0
Val internal duplicates:   0
Test internal duplicates:  0

--- Cross-Split Overlaps (Canonical Pairs) ---
Train AND Validation: 0
Train AND Test:       0
Validation AND Test:  0

--- Node-Level (Protein) Representation ---
Train Unique Proteins: 12323
Val Unique Proteins:   10231
Test Unique Proteins:  10278
Test Proteins in Train: 10278 / 10278 (100.0%) [Transductive Link Prediction]
Test Nove

In [18]:
# Step 5: ESM-2 150M Feature Extraction (Smart Check)
# Checks if embeddings exist locally or in Drive backup to avoid re-extracting
import os

emb_path = "data/processed/embeddings.pt"
drive_emb = "/content/drive/MyDrive/transgraph_ppi_esm150m/data_processed/embeddings.pt"

if os.path.exists(emb_path) and os.path.getsize(emb_path) > 1000000:
    size_mb = os.path.getsize(emb_path) / (1024 * 1024)
    print(f"Embeddings already exist at {emb_path} ({size_mb:.1f} MB). Skipping extraction.")
elif os.path.exists(drive_emb):
    print(f"Restoring pre-computed embeddings from Google Drive: {drive_emb}...")
    !mkdir -p data/processed
    !cp "$drive_emb" "$emb_path"
    print("Embeddings successfully restored from Drive!")
else:
    print("Extracting 640-d ESM-2 150M embeddings for all 12,323 proteins (takes ~25-30 mins on T4 GPU)...")
    !python src/data/feature_extraction.py --batch-size 16
    print("Feature extraction complete.")


Embeddings already exist at data/processed/embeddings.pt (18.5 MB). Skipping extraction.


In [19]:
# Step 6: Graph Construction (640-d ESM + 3 Topological features = 643-d nodes)
import os

graph_path = "data/processed/ppi_graph.pt"
drive_graph = "/content/drive/MyDrive/transgraph_ppi_esm150m/data_processed/ppi_graph.pt"
drive_map = "/content/drive/MyDrive/transgraph_ppi_esm150m/data_processed/ppi_graph_mapping.pt"

if os.path.exists(graph_path) and os.path.getsize(graph_path) > 1000000:
    print(f"PPI graph already exists at {graph_path}. Skipping.")
elif os.path.exists(drive_graph) and os.path.exists(drive_map):
    print("Restoring pre-built graph from Google Drive...")
    !mkdir -p data/processed
    !cp "$drive_graph" "$graph_path"
    !cp "$drive_map" "data/processed/ppi_graph_mapping.pt"
    print("Graph restored successfully from Drive.")
else:
    print("Building interaction graph and calculating topological features...")
    !python src/data/graph_construction.py

# Backup processed data to Drive if available
if os.path.exists("/content/drive/MyDrive"):
    !mkdir -p /content/drive/MyDrive/transgraph_ppi_esm150m/data_processed
    !cp -r data/processed/*.csv data/processed/*.pt /content/drive/MyDrive/transgraph_ppi_esm150m/data_processed/ 2>/dev/null || true
    print("Data backed up to Google Drive.")


PPI graph already exists at data/processed/ppi_graph.pt. Skipping.
Data backed up to Google Drive.


In [20]:
# Step 7: Train Sequence Model (ESM-2 150M Residual MLP)
import os
checkpoint_dir = "/content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints" if os.path.exists("/content/drive/MyDrive") else "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Training Sequence Model with early stopping (patience 10)...")
!python src/training/train_sequence_model.py --embedding_path data/processed/embeddings.pt --checkpoint_dir "{checkpoint_dir}" --ckpt_every 5


Training Sequence Model with early stopping (patience 10)...
Runtime: CUDA (Tesla T4)
Dataset: 161369 train / 20171 val samples
Feature dimension: 640 | batch size 128 | max epochs 30
/content/Majorproject/src/training/base_trainers.py:88: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  y_fit = torch.as_tensor(y_fit, dtype=torch.float32, device=device)
    resumed from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/sequence_checkpoint.pt at epoch 25
    [seq] epoch 26/30 train 0.0185 val 0.0426   (11.3s)
    [seq] epoch 27/30 train 0.0178 val 0.0430   (10.5s)
    [seq] early stopping at epoch 27

Sequence m

In [21]:
# Step 8: Train Graph Model (GraphSAGE Link Prediction & Platt Calibration)
print("Training GraphSAGE Model and calculating Platt probability calibration...")
!python src/training/train_graph_model.py --graph_path data/processed/ppi_graph.pt --checkpoint_dir "{checkpoint_dir}" --ckpt_every 5


Training GraphSAGE Model and calculating Platt probability calibration...
Runtime: CUDA (Tesla T4)
Training SAGE link predictor on cuda (full-batch)...
Link-prediction pairs: 161369 train / 20171 val | nodes 12323 | max epochs 100
/content/Majorproject/src/training/base_trainers.py:157: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  y_f = torch.as_tensor(y_fit, dtype=torch.float32, device=device)
    resumed from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/graph_checkpoint.pt at epoch 100

Graph model training complete in 0.0 min. Saved /content/Majorproject/models/graph_model_best.pth
GraphSAGE calibra

In [22]:
# Step 9: Train Random Forest Baseline Model
print("Fitting Random Forest Baseline on train.csv...")
!python src/training/train_random_forest.py


Fitting Random Forest Baseline on train.csv...
      TRANSGRAPH-PPI: TRADITIONAL RANDOM FOREST BASELINE TRAINING      
Configuration: n_estimators=100, max_depth=None, dry_run=False

[1/5] Loading ESM Embeddings & Biological Utilities...
  Loaded 12323 protein ESM embeddings.
  Loaded biological localization multi-hot encodings for 76 proteins.
  Initialized BiologicalManager for localization compatibility scoring.

[2/5] Constructing Feature Matrix for Training Data (train.csv ONLY)...
Extracting Train Features: 100% 161369/161369 [05:21<00:00, 501.93it/s]

[LEAKAGE CHECK & FEATURE AUDIT]
  Training Samples: 161369 (Positive: 80685, Negative: 80684)
  Feature Vector Shape: (161369, 2581) (Exact Feature Dimension: 2581)
  Missing Embedding Pairs Filtered: 0
  Zero NaNs: True, Zero Infs: True
  [VERIFIED] Fitted strictly on train.csv. val.csv and test.csv were NOT touched during fitting.

[3/5] Fitting RandomForestClassifier (100 trees, n_jobs=-1)...
[Parallel(n_jobs=-1)]: Using backend

In [23]:
# Step 10: Train 5-Fold OOF Stacking Ensemble (XGBoost)
print("Training 5-Fold Out-Of-Fold base models and XGBoost Stacking Ensemble...")
!python src/training/train_ensemble.py --checkpoint_dir "{checkpoint_dir}" --ckpt_every 5


Training 5-Fold Out-Of-Fold base models and XGBoost Stacking Ensemble...
Running Ensemble Training Pipeline on cuda...
Loading support files (embeddings, graph data, node mapping, bio features)...
Meta-Learner Training Dataset: 161369 samples from train.csv.

--- Leakage-free OOF predictions: 5 folds, max 30/100 epochs (seq/graph) with early stopping, device=cuda ---

Fold 1/5: loaded finished result from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/oof/fold1.npz

Fold 2/5: loaded finished result from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/oof/fold2.npz

Fold 3/5: loaded finished result from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/oof/fold3.npz

Fold 4/5: loaded finished result from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/oof/fold4.npz

Fold 5/5: loaded finished result from /content/drive/MyDrive/transgraph_ppi_esm150m/checkpoints/oof/fold5.npz

[VERIFIED] All 161369 training samples received exactly one leakage-free O

In [25]:
# Step 11: Comprehensive Held-Out Test Evaluation
# Evaluates all models on test.csv (20,172 pairs) and saves final_test_metrics.json
print("Evaluating models on held-out test split...")
!python src/evaluation/compare_models.py


Evaluating models on held-out test split...
Evaluating models on cuda...
Loaded 20171 validation samples and 20172 test samples.
Loading Sequence Model checkpoint (sequence_model_best.pth)...
Loading Graph Model checkpoint (graph_model_best.pth)...
Loaded XGBoost Meta-Learner checkpoint (ensemble_model.pkl).
Loaded Random Forest Baseline checkpoint (random_forest_baseline.pkl).

--- STEP 1: Selecting Decision Thresholds on Validation Set (val.csv ONLY) ---
Val Inference (Graph): 100% 5/5 [00:00<00:00, 339.93it/s]
Val Inference (RF Baseline): 100% 20171/20171 [00:39<00:00, 507.29it/s]
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    1.0s finished

[VALIDATION THRESHOLDS SELECTED]
  Sequence Model Optimal Threshold (val.csv): 0.4500
  Graph Model Optimal Threshold    (val.csv): 0.6100
  Random Forest Optimal Threshold  (val.csv): 0.4800
  Ens

In [26]:
# Step 12: External Benchmarks & Generalization Evaluation
print("--- 1. Cold-Start Evaluation ---")
!python scripts/evaluation/cold_start_eval.py

print("\n--- 2. SHS27k External Benchmark ---")
!python scripts/evaluation/external_benchmark_shs27k.py

print("\n--- 3. HuRI Independent Benchmark ---")
!python scripts/evaluation/external_benchmark_huri.py


--- 1. Cold-Start Evaluation ---
Loaded meta-learner from /content/Majorproject/models/ensemble_model.pkl
Loaded GraphSAGE calibrator from /content/Majorproject/models/graph_calibrator.json
Removing 400 proteins from the trained graph as novel (unseen-at-inference) proteins.
Pruned graph: 11923 nodes (was 12323), 152346 directed edges (was 161370).
Vectorized KNN verified against reference implementation on a probe case.
Eligible semi-cold-start test pairs (exactly one endpoint held out): 1152
Eligible pairs with embeddings present: 1152
Sequence model scored.
  graph scoring 0/1152
  graph scoring 200/1152
  graph scoring 400/1152
  graph scoring 600/1152
  graph scoring 800/1152
  graph scoring 1000/1152
Scored pairs: 1152
{
  "description": "Simulated semi-cold-start eval: one pair endpoint physically removed from the trained graph and reconstructed via the production insert_novel_node_knn() path; no retraining. See scripts/evaluation/cold_start_eval.py docstring for caveats before 

In [27]:
# Step 13: Display Metrics & Generate Audit Hashes
import json
import pandas as pd

metrics_file = "assets/evaluation/final_test_metrics.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        metrics_data = json.load(f)
    print("\n=================== FINAL TEST PERFORMANCE SUMMARY ===================")
    summary_df = pd.DataFrame(metrics_data).T
    cols = [c for c in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"] if c in summary_df.columns]
    print(summary_df[cols])

# Hashes and audits
!sha256sum data/processed/*.csv data/processed/*.pt models/*.pth models/*.pkl models/*.json > assets/evaluation/artifact_hashes.txt 2>/dev/null || true
!python scripts/evaluation/audit_measurements.py --out assets/evaluation/audit/audit_after_full.json
!pip freeze > requirements-lock.txt
print("Artifact hashes, audits, and requirements-lock.txt updated.")



=================== FINAL TEST PERFORMANCE SUMMARY ===================
Empty DataFrame
Columns: []
Index: [timestamp_utc, protocol, split_design, dataset_rows, models, checkpoints]
{
  "split_sizes": {
    "train": 161369,
    "val": 20171,
    "test": 20172
  },
  "negative_contamination": {
    "train": {
      "n_negatives": 80684,
      "contaminated_any_score": 0,
      "contaminated_ge400": 0,
      "contaminated_ge700": 0,
      "contamination_pct_any": 0.0
    },
    "val": {
      "n_negatives": 10086,
      "contaminated_any_score": 0,
      "contaminated_ge400": 0,
      "contaminated_ge700": 0,
      "contamination_pct_any": 0.0
    },
    "test": {
      "n_negatives": 10086,
      "contaminated_any_score": 0,
      "contaminated_ge400": 0,
      "contaminated_ge700": 0,
      "contamination_pct_any": 0.0
    },
    "ALL": {
      "n_negatives": 100856,
      "contaminated_any_score": 0,
      "contaminated_ge400": 0,
      "contaminated_ge700": 0,
      "contamination_pc

In [28]:
# Step 14: Package Trained Models & Artifacts
zip_target = "/content/drive/MyDrive/esm150m_results.zip" if os.path.exists("/content/drive/MyDrive") else "esm150m_results.zip"

!zip -r "{zip_target}" \
    models \
    assets/evaluation \
    data/processed/embeddings.pt \
    data/processed/ppi_graph.pt \
    data/processed/ppi_graph_mapping.pt \
    requirements-lock.txt

print(f"\nSUCCESS: All models and metrics packaged into: {zip_target}")


  adding: models/ (stored 0%)
  adding: models/ensemble_model.pkl (deflated 64%)
  adding: models/experiments/ (stored 0%)
  adding: models/experiments/features_cache.npz (deflated 43%)
  adding: models/graph_calibrator.json (deflated 8%)
  adding: models/graph_model_best.pth (deflated 8%)
  adding: models/sequence_model_best.pth (deflated 7%)
  adding: models/random_forest_baseline.pkl (deflated 79%)
  adding: assets/evaluation/ (stored 0%)
  adding: assets/evaluation/confusion_matrix_sequence-only.png (deflated 15%)
  adding: assets/evaluation/external_benchmark_shs27k.json (deflated 48%)
  adding: assets/evaluation/final_test_metrics.json (deflated 62%)
  adding: assets/evaluation/threshold_analysis_ensemble.png (deflated 6%)
  adding: assets/evaluation/cold_start_eval_seed7.json (deflated 47%)
  adding: assets/evaluation/graph_calibration.json (deflated 49%)
  adding: assets/evaluation/confusion_matrix_graph-only.png (deflated 14%)
  adding: assets/evaluation/threshold_analysis_gra